# 04 — Plan-and-Execute

**Model:** DeepSeek-V3 via Nebius AI Studio  
**Pattern:** Decoupled Planner + Executor  
**Difficulty:** ⭐⭐⭐☆☆

---

## The Problem

ReAct agents are *reactive* — they decide the next action only after seeing the previous observation. For simple questions this is fine. For complex, multi-step goals it's a liability: the agent can lose sight of the big picture, get distracted by intermediate results, or fail to parallelize independent sub-tasks.

**Plan-and-Execute** separates strategic thinking from execution:
- The **Planner** receives the goal and produces a numbered list of steps — *before* any tools are called.
- The **Executor** carries out each step in sequence, feeding results forward.
- An optional **Replanner** can revise the remaining plan if a step fails or reveals new information.

## Architecture

```
┌──────────────────────────────────────────────────────────┐
│                  Plan-and-Execute Graph                   │
│                                                           │
│  ┌──────────┐    ┌──────────┐    ┌──────────────────┐   │
│  │ Planner  │───▶│ Executor │───▶│  More steps?     │   │
│  └──────────┘    └────▲─────┘    └──────┬───────────┘   │
│                        │           yes  │    no → END    │
│                        └───────────────┘                 │
│                        (next step)                       │
└──────────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict, List, Optional
from tavily import TavilyClient
import math

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("Setup complete.")

## State & Tools

In [ ]:
class PlanExecuteState(TypedDict):
    goal: str
    plan: List[str]          # Ordered list of steps from planner
    current_step_index: int
    step_results: List[str]  # Results accumulated from executed steps
    final_summary: Optional[str]


@tool
def search(query: str) -> str:
    """Search the web for current information on any topic."""
    results = tavily.search(query=query, max_results=2)
    return "\n".join(r["content"] for r in results.get("results", []))


@tool
def compute(expression: str) -> str:
    """Evaluate a mathematical expression. E.g. compute("1_000_000 / 365")"""
    ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    ns["__builtins__"] = {}
    try:
        return str(eval(expression.replace("_", ""), ns))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"


tools = [search, compute]
llm_with_tools = llm.bind_tools(tools)

## Planner Node

The planner must return a clean numbered list. We ask for structured JSON to make parsing reliable.

In [ ]:
PLANNER_PROMPT = """You are a planning agent. Given a goal, break it into a clear, ordered list of concrete steps.

Rules:
- Each step must be independently executable by a research/calculation agent.
- Keep steps specific. "Search for X" is good. "Research everything" is not.
- Maximum 6 steps.
- Return ONLY a JSON array of strings: ["Step 1 description", "Step 2 description", ...]
"""


def planner_node(state: PlanExecuteState) -> dict:
    """Generate an ordered execution plan for the goal."""
    print(f"\n[Planner] Planning for: '{state['goal']}'")

    response = llm.invoke([
        SystemMessage(content=PLANNER_PROMPT),
        HumanMessage(content=f"Goal: {state['goal']}")
    ])

    # Extract JSON from response
    raw = response.content.strip()
    json_match = raw[raw.find("["):raw.rfind("]")+1]
    try:
        plan = json.loads(json_match)
    except json.JSONDecodeError:
        # Fallback: treat each line as a step
        plan = [line.strip() for line in raw.split("\n") if line.strip()]

    print(f"[Planner] Generated {len(plan)}-step plan:")
    for i, step in enumerate(plan, 1):
        print(f"  {i}. {step}")

    return {"plan": plan, "current_step_index": 0, "step_results": []}

## Executor Node

The executor receives one step at a time and uses tools to complete it. It has full context of prior step results.

In [ ]:
EXECUTOR_PROMPT = """You are a research and calculation agent. You will be given:
1. The overall goal
2. Results from steps you've already completed
3. The current step to execute

Use your tools (search, compute) to complete the current step. Return a concise, factual answer.
If the step doesn't require a tool, answer from reasoning alone.
"""


def executor_node(state: PlanExecuteState) -> dict:
    """Execute the current step of the plan."""
    idx = state["current_step_index"]
    step = state["plan"][idx]

    print(f"\n[Executor] Step {idx + 1}/{len(state['plan'])}: {step}")

    # Build context from prior results
    prior_context = ""
    for i, res in enumerate(state["step_results"]):
        prior_context += f"Step {i+1} result: {res}\n"

    user_msg = (
        f"Overall goal: {state['goal']}\n\n"
        f"{prior_context}\n"
        f"Current step to execute: {step}"
    )

    messages = [
        SystemMessage(content=EXECUTOR_PROMPT),
        HumanMessage(content=user_msg)
    ]

    # Agentic tool loop for this single step
    for _ in range(5):  # max 5 tool calls per step
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not (hasattr(response, "tool_calls") and response.tool_calls):
            break

        # Execute tools
        from langchain_core.messages import ToolMessage
        for tc in response.tool_calls:
            fn = {"search": search, "compute": compute}[tc["name"]]
            tool_result = fn.invoke(tc["args"])
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tc["id"]))

    result_text = response.content
    print(f"   Result: {result_text[:150]}...")

    new_results = state["step_results"] + [result_text]
    return {
        "step_results": new_results,
        "current_step_index": idx + 1
    }


def summarizer_node(state: PlanExecuteState) -> dict:
    """Synthesize all step results into a final answer."""
    print("\n[Summarizer] Synthesizing results...")

    results_text = "\n".join(
        f"Step {i+1}: {r}" for i, r in enumerate(state["step_results"])
    )

    response = llm.invoke([
        SystemMessage(content="You synthesize research results into clear, complete answers."),
        HumanMessage(content=(
            f"Original goal: {state['goal']}\n\n"
            f"Completed steps:\n{results_text}\n\n"
            "Write a comprehensive final answer:"
        ))
    ])

    return {"final_summary": response.content}


def check_steps_remaining(state: PlanExecuteState) -> str:
    if state["current_step_index"] >= len(state["plan"]):
        return "summarize"
    return "execute"

## Build the Graph

In [ ]:
builder = StateGraph(PlanExecuteState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("summarizer", summarizer_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges(
    "executor",
    check_steps_remaining,
    {"execute": "executor", "summarize": "summarizer"}
)
builder.add_edge("summarizer", END)

graph = builder.compile()
print("Plan-and-Execute graph compiled.")

## Live Demo

**Scenario:** A goal that requires multiple distinct research tasks — ideal for demonstrating why upfront planning beats reactive step-by-step.

In [ ]:
goal = (
    "Compare the GDP per capita of Egypt and the UAE as of the latest available data. "
    "Calculate the ratio between them, and explain what it tells us about economic development in each country."
)

result = graph.invoke({
    "goal": goal,
    "plan": [],
    "current_step_index": 0,
    "step_results": [],
    "final_summary": None
})

In [ ]:
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_summary"])

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Decoupled planning** | Planner runs once upfront; executor never makes strategic decisions |
| **State as accumulator** | `step_results` builds context for each subsequent step |
| **Inner tool loop** | Each executor step can call multiple tools before returning |
| **Clean separation** | Easy to swap planner model (reasoning) vs executor model (speed) |

## What's Next

**Notebook 05** scales this to a full multi-agent team with a Supervisor who dynamically dispatches tasks to specialist workers — not a fixed sequence of steps, but a flexible routing system.